## 00. Libraries, Programming Models, and Python-C++ Interoperability

### Table of Contents

1. [Motivation](#sec1)
2. [The problem: a 1D bump pulse](#sec2)
3. [Why this small problem](#sec3)
4. [What we are going to try:](#sec4)
5. [How to read these notebooks](#sec5)

### <a id="sec1"></a>1. Motivation

We have seen powerful array-oriented libraries in Python capable of scaling with increasing problem complexity and data volumes. Now let's tour another aspect of the HPC Python landscape:
Reaching for a different programming model, library, or dropping into a C/C++ kernel to integrate with an existing C/C++ codebase you already maintain or depend on. For the latter, some familiarity with C/C++ is required, but if you're new to it, you can still follow along as the programming syntax is not the primary focus.


### <a id="sec2"></a>2. The problem: a 1D bump pulse

During this hour, we solve the same problem with different approaches, each aiming to leverage the maximum throughput from the available hardware.

![alt text](../assets/Shallow_water_waves.gif)

**1D Shallow Water Equations**

The SWE are a set of hyperbolic partial differential equations that describe the flow below a pressure surface in a fluid. 

In its conservative form, this equation consists of two variables per cell:
water height `h` and momentum `hu`.

The initial condition is a **Gaussian-bump pulse**: a small mound of
water (10% above a 1 m baseline) sitting at rest in
the middle of the channel at $t = 0$.

Gravity makes it split into two
counter-propagating wave packets travelling outward at $c = \sqrt{g\,h_0}
\approx 3.13$ m/s.

The goal of our solution is to simulate what the water surface looks like
after $N$ time-steps. A complete overview of the problem spec lives in `tutorials/python-hpc-libraries-swe/docs/PROBLEM.md`. For simplicity, we consider the 1D conservative form of this problem.

Let's visualise the initial condition and the solution after 1000 steps with a reference NumPy kernel. `swe_core.py` provides a reference implementation that we will look at in the next notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import swe_core

N, L = 4096, 10.0
dx = L / N
h0, hu0 = swe_core.bump_ic(N, L=L, h0=1.0, amplitude=0.1, sigma=0.5)
h, hu = h0.copy(), hu0.copy()
dt = swe_core.fixed_dt(1.1, dx)
for _ in range(1000):
    swe_core.apply_bc_reflective(h, hu)
    h, hu = swe_core.step_numpy(h, hu, dx, dt)

xs = (np.arange(N) + 0.5) * dx
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 3.6))
axL.plot(xs, h0[1:-1], color='#888', label='t=0 (IC)', linewidth=2)
axL.plot(xs, h[1:-1], color='#27a', label='t after 1000 steps')
axL.set_xlabel('x [m]'); axL.set_ylabel('h [m]')
axL.set_title('Water height — bump pulse, 1D'); axL.legend(); axL.grid(alpha=0.3)

axR.plot(xs, hu0[1:-1], color='#888', label='t=0 (IC)', linewidth=2)
axR.plot(xs, hu[1:-1], color='#c33', label='t after 1000 steps')
axR.set_xlabel('x [m]'); axR.set_ylabel('hu [m²/s]')
axR.set_title('Momentum hu'); axR.legend(); axR.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The plot above shows the initial bump and
the surface after 1000 steps, following the equations we described earlier.

The bump splits into a left-moving and a right-moving wave
packet, each carrying about half the original amplitude.

Let's now render everything in between as an animation. To keep the embedded animation compact, we render on a
coarser 256-cell grid over a wider 20 m channel; across the tutorial we will scale to higher resolutions.

In [ ]:
# Animation of the bump-pulse evolution. We use a coarser grid (N=256) and
# capture h every 20 steps so the embedded JS animation stays under ~2 MB.
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

N_anim    = 256
L_anim    = 20
dx_anim   = L_anim / N_anim
SKIP      = 40      # render every SKIP'th step
N_STEPS_A = 2500

h, hu  = swe_core.bump_ic(N_anim, L=L_anim)
dt_a   = swe_core.fixed_dt(1.1, dx_anim)
frames = [(0, h[1:-1].copy())]
for step in range(1, N_STEPS_A + 1):
    swe_core.apply_bc_reflective(h, hu)
    h, hu = swe_core.step_numpy(h, hu, dx_anim, dt_a)
    if step % SKIP == 0:
        frames.append((step, h[1:-1].copy()))

xs = (np.arange(N_anim) + 0.5) * dx_anim
fig, ax = plt.subplots(figsize=(8, 3.6))
line, = ax.plot(xs, frames[0][1], color='#27a', linewidth=2)
ax.set_xlim(0, L_anim); ax.set_ylim(0.93, 1.13)
ax.set_xlabel('x  [m]'); ax.set_ylabel('h  [m]')
ax.set_title('1D bump pulse — water height over time')
ax.grid(alpha=0.3)
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes,
                    fontsize=10, verticalalignment='top')

def _update(idx):
    step_idx, h_frame = frames[idx]
    line.set_ydata(h_frame)
    time_text.set_text(f'step {step_idx:4d}   t = {step_idx*dt_a:.3f} s')
    return line, time_text

anim = FuncAnimation(fig, _update, frames=len(frames),
                     interval=80, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

### <a id="sec3"></a>3. Why this small problem

A 1D bump pulse with two conserved variables is a relatively small non-trivial PDE which can be expressed in ~30 lines of vectorised NumPy. But it is complex enough to surface certain computing challenges:

- Memory-bound stencil at large `N`
- Per-element nonlinearity (`hu²/h`, `sqrt(g h)`), which goes beyond the classic `saxpy` example

The results can also be compared across tools to a fixed tolerance after a set number of time-steps from the initial condition, which is useful for validation.

### <a id="sec4"></a>4. What we are going to try:

| NB | Tool | Programming-model hook |
|---|---|---|
| 01 | NumPy | Vectorised arrays: Baseline reference |
| 02 | **JAX** | JIT compilation + whole-loop fusion offloaded to device |
| 03 | **PyOMP** | OpenMP parallelisation model in Python |
| 04 | **nanobind** | Handwritten Python bindings for compiled C++ kernels |
| 05 | **CppJIT** | JIT compiled C++ with automatic bindings|

We end with `06__synthesis.ipynb`: a decision matrix and the achieved-throughput plot across the different notebooks in `timings.json`.

### <a id="sec5"></a>5. How to read these notebooks

Each numbered exercise (02-06) comes as two files:

- `NN__topic__subtopic.ipynb`: the primary notebook we run live during the tutorial, with `# TODO:` cells whose body is a `...` placeholder.
- `solutions/NN__topic__subtopic__SOLUTION.ipynb`: notebooks complete with solution, for self-paced review.

Within each exercise notebook:

- A **Quick Docs** sub-block precedes each TODO cell, listing the key APIs you need.
- Optional **`# BONUS TASK:`** cells explore deeper solutions if you have time.

Let us start where we left off: using NumPy as a baseline `01__swe_core__reference_solver.ipynb`